In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from pathlib import Path
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

import torchvision
from torchvision.models.detection.retinanet import RetinaNet_ResNet50_FPN_Weights
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from matplotlib import pyplot as plt

# Configuration
TRAIN_IMAGES_DIR = r"C:\\ARI3129_work\\Datasets\\COCO-based_COCO\\images\\train"
VAL_IMAGES_DIR   = r"C:\\ARI3129_work\\Datasets\\COCO-based_COCO\\images\\val"
TRAIN_JSON       = r"C:\\ARI3129_work\\Datasets\\COCO-based_COCO\\annotations\\train.json"
VAL_JSON         = r"C:\\ARI3129_work\\Datasets\\COCO-based_COCO\\annotations\\val.json"
TEST_IMAGES_DIR  = r"C:\ARI3129_work\Datasets\COCO-based_COCO\images\test"
TEST_JSON        = r"C:\ARI3129_work\Datasets\COCO-based_COCO\annotations\test.json"

BATCH_SIZE = 1
EPOCHS = 25
LR = 1e-4
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" 

# Early stopping configuration
PATIENCE = 10         # stop if no val AP improvement for this many epochs
MIN_DELTA = 1e-4      # minimum AP improvement to count as "better"
EVAL_THRESH = 0.001   # COCO-style low threshold for fair evaluation during training

print(DEVICE)


cpu


In [3]:

# Dataset
class CocoRetinaDataset(Dataset):
    """
    Returns:
      image: FloatTensor [3,H,W] in 0..1
      target: dict with keys required by torchvision detection models:
        - boxes: FloatTensor [N,4] in XYXY
        - labels: Int64Tensor [N] (1..K)  (IMPORTANT: 0 is background)
        - image_id: Int64Tensor [1]
        - area: FloatTensor [N]
        - iscrowd: Int64Tensor [N]
    """
    def __init__(self, images_dir: str, ann_json: str):
        self.images_dir = Path(images_dir)
        self.coco = COCO(ann_json)
        self.img_ids = list(self.coco.imgs.keys())

        cats = self.coco.loadCats(self.coco.getCatIds())
        cats = sorted(cats, key=lambda x: x["id"])
        self.cat_id_to_label = {c["id"]: i + 1 for i, c in enumerate(cats)}  # 1..K
        self.label_to_cat_id = {v: k for k, v in self.cat_id_to_label.items()}
        self.cat_id_to_name = {c["id"]: c["name"] for c in cats}
        self.label_to_name = {self.cat_id_to_label[c["id"]]: c["name"] for c in cats}
        self.num_classes = len(self.cat_id_to_label) + 1  # + background

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx: int):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = self.images_dir / img_info["file_name"]

        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        ann_ids = self.coco.getAnnIds(imgIds=[img_id], iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, area, iscrowd = [], [], [], []

        for a in anns:
            if a.get("iscrowd", 0) == 1:
                continue
            x, y, bw, bh = a["bbox"]
            if bw <= 1 or bh <= 1:
                continue

            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(w, x + bw)
            y2 = min(h, y + bh)
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_label[a["category_id"]])
            area.append((x2 - x1) * (y2 - y1))
            iscrowd.append(0)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            area = torch.tensor(area, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id], dtype=torch.int64),
            "area": area,
            "iscrowd": iscrowd,
        }
        return img_t, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

# COCO evaluation
@torch.no_grad()
def evaluate_coco(model, dataset: CocoRetinaDataset, ann_json: str, score_thresh: float = 0.05):
    model.eval()
    coco_gt = COCO(ann_json)
    results = []

    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

    for images, targets in loader:
        images = [images[0].to(DEVICE)]
        img_id = int(targets[0]["image_id"].item())

        outputs = model(images)[0]  # dict that includes boxes, labels, scores

        boxes = outputs["boxes"].detach().cpu().numpy()
        labels = outputs["labels"].detach().cpu().numpy().astype(int)
        scores = outputs["scores"].detach().cpu().numpy()

        for (x1, y1, x2, y2), lab, sc in zip(boxes, labels, scores):
            if sc < score_thresh:
                continue
            if int(lab) not in dataset.label_to_cat_id:
                continue
            cat_id = dataset.label_to_cat_id[int(lab)]

            w = float(x2 - x1)
            h = float(y2 - y1)

            results.append({
                "image_id": img_id,
                "category_id": int(cat_id),
                "bbox": [float(x1), float(y1), w, h],  # as COCO wants XYWH
                "score": float(sc),
            })

    if not results:
        print("No detections to evaluate.")
        return {}

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return {
        "AP_50_95": float(coco_eval.stats[0]),
        "AP_50": float(coco_eval.stats[1]),
        "AP_75": float(coco_eval.stats[2]),
    }

# Helper function
def get_first_conv_in_channels(module: torch.nn.Module) -> int:
    for m in module.modules():
        if isinstance(m, torch.nn.Conv2d):
            return m.in_channels
    raise RuntimeError("Could not find a Conv2d inside the classification head.")


In [ ]:
# Training
def main():
    train_ds = CocoRetinaDataset(TRAIN_IMAGES_DIR, TRAIN_JSON)
    val_ds = CocoRetinaDataset(VAL_IMAGES_DIR, VAL_JSON)

    print("Num classes (including background):", train_ds.num_classes)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn
    )

    # RetinaNet ResNet50-FPN pretrained on COCO
    weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT
    model = torchvision.models.detection.retinanet_resnet50_fpn(weights=weights)

    num_classes = train_ds.num_classes  # includes background (0)
    num_anchors = model.head.classification_head.num_anchors
    in_channels = get_first_conv_in_channels(model.head.classification_head)

    model.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
    )

    model.to(DEVICE)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR)

    # Early stopping state
    best_ap = -1.0
    epochs_no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0

        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            running += float(loss.item())

        avg_loss = running / max(1, len(train_loader))
        print(f"Epoch {epoch}/{EPOCHS} - train loss: {avg_loss:.4f}")

        stats = evaluate_coco(model, val_ds, VAL_JSON, score_thresh=EVAL_THRESH)
        print("Val stats:", stats)

        # Early stopping check 
        current_ap = stats.get("AP_50_95", None)
        if current_ap is None:
            print("Warning: evaluation returned no AP_50_95; skipping early stopping check this epoch.")
            continue

        if current_ap > best_ap + MIN_DELTA:
            best_ap = current_ap
            epochs_no_improve = 0
            torch.save(model.state_dict(), "retinanet_best.pth")
            print(f"New best model saved: retinanet_best.pth (AP_50_95={best_ap:.4f})")
        else:
            epochs_no_improve += 1
            print(f"No improvement: {epochs_no_improve}/{PATIENCE} (best AP_50_95={best_ap:.4f})")

        if epochs_no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

    # Save last epoch weights too, in case of crash to not lose final state
    torch.save(model.state_dict(), "retinanet_custom.pth")
    print("Saved: retinanet_custom.pth")
    print("Best weights (if improved): retinanet_best.pth")

if __name__ == "__main__":
    main()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Num classes (including background): 7
Epoch 1/25 - train loss: 0.9574
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.24s).
Accumulating evaluation results...
DONE (t=0.09s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.122
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.174
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.143
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=10

In [5]:
test_ds = CocoRetinaDataset(TEST_IMAGES_DIR, TEST_JSON)

# Recreate the model
weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT
model = torchvision.models.detection.retinanet_resnet50_fpn(weights=weights)

num_classes = test_ds.num_classes
num_anchors = model.head.classification_head.num_anchors
in_channels = get_first_conv_in_channels(model.head.classification_head)

model.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
    in_channels=in_channels, num_anchors=num_anchors, num_classes=num_classes
)

#Load trained weights
state = torch.load("retinanet_best.pth", map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE)

model.score_thresh = 0.001     # higher confidence, fewer weak boxes
model.nms_thresh = 0.4      # suppress overlapping duplicates
model.detections_per_img = 100     # cap max detections

model.eval()





loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


C:\Users\User\AppData\Local\Temp\ipykernel_42004\2918160273.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load("retinanet_best.pth", map_location=DEVICE

RetinaNet(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d(256, eps=0.0)


In [7]:
test_stats = evaluate_coco(model, test_ds, TEST_JSON, score_thresh=0.05)
print("TEST stats:", test_stats)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.671
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.738
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.724
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.034
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.712
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.724
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.731
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [ ]:
import random
import numpy as np

@torch.no_grad()
def predict_and_draw(model, dataset, idx=None, score_thresh=0.5, topk=5):
    model.eval()
    if idx is None:
        idx = random.randint(0, len(dataset) - 1)

    img_t, target = dataset[idx]

    # Convert tensor to RGB image
    img = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()

    # Run inference
    out = model([img_t.to(DEVICE)])[0]

    boxes  = out["boxes"].detach().cpu().numpy()
    labels = out["labels"].detach().cpu().numpy().astype(int)
    scores = out["scores"].detach().cpu().numpy()

    # filter by score
    keep = scores >= score_thresh
    boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

    # keep top-K highest scores
    if len(scores) > topk:
        order = np.argsort(-scores)[:topk]
        boxes, labels, scores = boxes[order], labels[order], scores[order]

    for (x1, y1, x2, y2), lab, sc in zip(boxes, labels, scores):
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        name = dataset.label_to_name.get(int(lab), str(int(lab)))
        text = f"{name} {sc:.2f}"

        # Output with text
        cv2.putText(img, text, (x1, max(0, y1-12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 3, (0,0,0), 4)
        cv2.putText(img, text, (x1, max(0, y1-12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 3, (0,255,0), 3)

    return img



# Visualization friendly metrics 
model.score_thresh = 0.4
model.nms_thresh = 0.5
model.detections_per_img = 5

# show predictions from test set
val_ds = CocoRetinaDataset(TEST_IMAGES_DIR, TEST_JSON)


OUT_DIR = "2a_results"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DIR = "2a_results"
os.makedirs(OUT_DIR, exist_ok=True)

# Run and save predictions
for i in range(len(val_ds)):
    img_vis = predict_and_draw(
        model,
        val_ds,
        idx=i,
        score_thresh=0.1
    )

    # Convert RGB -> BGR for OpenCV saving
    img_bgr = cv2.cvtColor(img_vis, cv2.COLOR_RGB2BGR)

    out_path = os.path.join(OUT_DIR, f"image_{i:03d}.jpg")
    cv2.imwrite(out_path, img_bgr)


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


'# Output Test images\nfor i in range(110):\n    img_vis = predict_and_draw(model, val_ds, idx=i, score_thresh=0.1)\n    plt.figure(figsize=(8,8))\n    plt.imshow(img_vis)\n    plt.title(f"Image {i}")\n    plt.axis("off")\n    plt.show()'

In [10]:
test_ds = CocoRetinaDataset(TEST_IMAGES_DIR, TEST_JSON)

# Used to log statistics on test set detections
@torch.no_grad()
def log_test_set_statistics(
    model,
    dataset,
    device,
    score_thresh=0.05
):
    model.eval()

    total_images = len(dataset)
    total_detections = 0
    class_counts = defaultdict(int)

    for idx in range(total_images):
        img_t, _ = dataset[idx]
        outputs = model([img_t.to(device)])[0]

        scores = outputs["scores"].detach().cpu()
        labels = outputs["labels"].detach().cpu()

        keep = scores >= score_thresh
        labels = labels[keep]

        total_detections += len(labels)

        for lab in labels:
            class_counts[int(lab)] += 1

    mean_per_image = total_detections / total_images if total_images > 0 else 0.0

    # Convert class ids to names (if available)
    named_counts = {}
    for lab, cnt in class_counts.items():
        name = dataset.label_to_name.get(lab, str(lab))
        named_counts[name] = cnt

    return {
        "total_images": total_images,
        "total_detections": total_detections,
        "mean_per_image": mean_per_image,
        "class_distribution": dict(named_counts),
    }


# Ensuring less multi-detection on one sign
model.score_thresh = 0.4
model.nms_thresh = 0.5
model.detections_per_img = 5 # Max in dataset was 4 in one image


stats = log_test_set_statistics(
    model,
    test_ds,
    DEVICE,
    score_thresh=0.05
)

print("Total test images:", stats["total_images"])
print("Total traffic signs detected:", stats["total_detections"])
print("Mean signs per image:", f'{stats["mean_per_image"]:.2f}')

print("\nDistribution of detected traffic sign types:")
for name, count in sorted(stats["class_distribution"].items(), key=lambda x: -x[1]):
    print(f"{name:30s} {count}")



loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Total test images: 110
Total traffic signs detected: 128
Mean signs per image: 1.16

Distribution of detected traffic sign types:
No Entry (One Way)             27
No Through Road (T-Sign)       26
Blind-Spot Mirror (Convex)     22
Roundabout Ahead               20
Stop                           18
Pedestrian Crossing            15
